# Tutorial 03: Inverse Kinematics

<p align="center">
  <img src="RRRP_SCARA.png" alt="RRRP SCARA Robot" width="500"/>
</p>

<p align="center"><i>
Source: <b>MODERN ROBOTICS: MECHANICS, PLANNING, AND CONTROL</b><br/>
Figure 4.12: An RRRP SCARA robot for performing pick-and-place operations.
</i></p>

---

**Prerequisites**: [Tutorial 01: Building Your First Robot](01_build_robot.ipynb), [Tutorial 02: Forward Kinematics](02_forward_kinematics.ipynb)

In this tutorial, we explore **Inverse Kinematics (IK)** - the problem of finding joint configurations that achieve a desired end-effector position and orientation.

### What is Inverse Kinematics?

| Forward Kinematics (FK) | Inverse Kinematics (IK) |
|------------------------|-------------------------|
| **Input**: Joint angles θ | **Input**: Desired end-effector pose |
| **Output**: End-effector pose | **Output**: Joint angles θ |
| **Method**: Direct computation | **Method**: Solve equations (analytical or numerical) |
| **Solution**: Unique | **Solution**: May have 0, 1, or multiple solutions |

### Why is IK Important?

Inverse kinematics is essential for:
- **Task-space control**: Move end-effector to a target position
- **Motion planning**: Plan paths for pick-and-place operations
- **Teleoperation**: Control robot using desired hand positions
- **Animation**: Generate realistic robot motions

### IK Challenges

1. **Multiple Solutions**: A target position may be reachable with different joint configurations (e.g., elbow left vs elbow right for SCARA)
2. **No Solution**: Target may be outside the robot's workspace
3. **Singularities**: Configurations where the robot loses a degree of freedom
4. **Numerical Instability**: Near singularities or workspace boundaries

### Solution Methods

We will cover two main approaches:

1. **Analytical (Closed-Form) Solution**: Derive explicit formulas for joint angles
   - Exact and fast
   - Only possible for simple kinematic structures
   
2. **Numerical Solution**: Iterative optimization using Jacobian
   - Works for any robot structure
   - May converge to local minima


## Setup and Imports

We import the same libraries as in the FK tutorial, plus Newton's IK module for numerical solutions.


In [1]:
import newton
import newton.ik  # Use newton.ik.* for clarity
import warp as wp
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from tqdm.notebook import trange
import os

# Initialize Warp
wp.init()

# Set NumPy print options for cleaner output
np.set_printoptions(precision=6, suppress=True, linewidth=100)


Warp 1.11.0.dev20251123 initialized:
   Git commit: 8b8f0b85ca54c0026574f834764e26615056aef6
   CUDA Toolkit 12.8, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA L40S" (44 GiB, sm_89, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.11.0.dev20251123


## Robot Parameters

We use the same RRRP SCARA robot parameters from Tutorial 02:
- **L₀ (base height)**: 0.10 m  
- **L₁ (first arm)**: 0.30 m
- **L₂ (second arm)**: 0.20 m
- **Prismatic stroke**: 0.0 to 0.25 m


In [ ]:
# Robot dimensions (meters)
L0 = 0.10            # Base height (height to first joint)
L1 = 0.30            # Length of first horizontal arm
L2 = 0.20            # Length of second horizontal arm

# Joint limits (expanded for better workspace coverage)
JOINT_LIMITS = np.array([
    [-3*np.pi/4, 3*np.pi/4],   # θ₁: [-135°, 135°]
    [-np.pi/2,   np.pi/2],     # θ₂: [-90°, 90°]
    [-2*np.pi/3, np.pi],       # θ₃: [-120°, 180°]
    [0.0,        0.25]         # θ₄: [0, 0.25m] (prismatic)
])

# Workspace limits
R_MAX = L1 + L2      # Maximum reach: 0.50 m
R_MIN = abs(L1 - L2) # Minimum reach: 0.10 m
Z_MIN = L0 + JOINT_LIMITS[3, 0]  # 0.60 m
Z_MAX = L0 + JOINT_LIMITS[3, 1]  # 0.85 m

print("Robot Parameters:")
print(f"  L₁ = {L1} m, L₂ = {L2} m")
print(f"\nWorkspace (reachable region):")
print(f"  Radial: {R_MIN:.2f} m ≤ r ≤ {R_MAX:.2f} m")
print(f"  Height: {Z_MIN:.2f} m ≤ z ≤ {Z_MAX:.2f} m")


Robot Parameters:
  L₁ = 0.3 m, L₂ = 0.2 m

Workspace (reachable region):
  Radial: 0.10 m ≤ r ≤ 0.50 m
  Height: 0.10 m ≤ z ≤ 0.35 m


## Forward Kinematics (Review)

From Tutorial 02, the forward kinematics for our SCARA robot:

$$x = \ell_1 \cos(\theta_1) + \ell_2 \cos(\theta_1 + \theta_2)$$
$$y = \ell_1 \sin(\theta_1) + \ell_2 \sin(\theta_1 + \theta_2)$$
$$z = \ell_0 + d_4$$

The end-effector orientation about Z-axis is: $\phi = \theta_1 + \theta_2 + \theta_3$


In [3]:
def fk_scara(q):
    """
    Forward kinematics for RRRP SCARA robot.
    
    Parameters:
        q: array [θ₁, θ₂, θ₃, θ₄] - joint values
    
    Returns:
        pos: array [x, y, z] - end-effector position
        phi: float - end-effector orientation (rotation about Z)
    """
    theta1, theta2, theta3, theta4 = q
    
    # Position
    x = L1 * np.cos(theta1) + L2 * np.cos(theta1 + theta2)
    y = L1 * np.sin(theta1) + L2 * np.sin(theta1 + theta2)
    z = L0 + theta4
    
    # Orientation (rotation about Z-axis)
    phi = theta1 + theta2 + theta3
    
    return np.array([x, y, z]), phi


# Test FK
q_test = np.array([0.0, 0.0, 0.0, 0.1])
pos, phi = fk_scara(q_test)
print(f"FK Test: q = {q_test}")
print(f"  Position: ({pos[0]:.4f}, {pos[1]:.4f}, {pos[2]:.4f})")
print(f"  Orientation: {np.degrees(phi):.1f}°")


FK Test: q = [0.  0.  0.  0.1]
  Position: (0.5000, 0.0000, 0.2000)
  Orientation: 0.0°


## Part 1: Analytical Inverse Kinematics

### The IK Problem for SCARA

Given a desired end-effector position $(x_d, y_d, z_d)$ and orientation $\phi_d$, find joint values $(\theta_1, \theta_2, \theta_3, \theta_4)$.

The SCARA structure allows us to **decouple** the problem:

1. **Vertical (Z-axis)**: Solved independently by the prismatic joint θ₄
2. **Horizontal (X-Y plane)**: Solved by the 2R planar arm (θ₁, θ₂)
3. **Orientation**: Solved by the wrist joint θ₃

### Step 1: Solve for θ₄ (Prismatic Joint)

From $z = h_{table} + \ell_0 + \theta_4$:

$$\theta_4 = z_d - h_{table} - \ell_0$$

**Validity check**: $0 \leq \theta_4 \leq 0.25$ m

### Step 2: Solve for θ₂ (Elbow Angle)

Using the law of cosines on the 2R arm:

$$r^2 = x_d^2 + y_d^2 = \ell_1^2 + \ell_2^2 + 2\ell_1\ell_2\cos(\theta_2)$$

Solving for θ₂:

$$\cos(\theta_2) = \frac{x_d^2 + y_d^2 - \ell_1^2 - \ell_2^2}{2\ell_1\ell_2}$$

$$\theta_2 = \pm \arccos\left(\frac{x_d^2 + y_d^2 - \ell_1^2 - \ell_2^2}{2\ell_1\ell_2}\right)$$

**Note**: The ± gives us **two solutions**: "elbow right" (+θ₂) and "elbow left" (-θ₂)

### Step 3: Solve for θ₁ (Shoulder Angle)

Using geometry of the 2R arm:

$$\theta_1 = \arctan2(y_d, x_d) - \arctan2(\ell_2\sin\theta_2, \ell_1 + \ell_2\cos\theta_2)$$

### Step 4: Solve for θ₃ (Wrist Angle)

From the orientation constraint $\phi_d = \theta_1 + \theta_2 + \theta_3$:

$$\theta_3 = \phi_d - \theta_1 - \theta_2$$


### Implementation of Analytical IK

The function below implements the closed-form IK solution. It returns both "elbow left" and "elbow right" configurations when available, or returns `None` if the target is unreachable.


In [4]:
def ik_scara_analytical(target_pos, target_phi=0.0, elbow_right=True):
    """
    Analytical inverse kinematics for RRRP SCARA robot.
    
    Parameters:
        target_pos: array [x, y, z] - desired end-effector position (m)
        target_phi: float - desired orientation about Z-axis (rad)
        elbow_right: bool - select elbow configuration (True=right/+θ₂, False=left/-θ₂)
    
    Returns:
        q: array [θ₁, θ₂, θ₃, θ₄] or None if unreachable
        info: dict with status information
    """
    x_d, y_d, z_d = target_pos
    
    info = {'reachable': True, 'message': 'Success'}
    
    # Step 1: Solve for θ₄ (prismatic joint)
    theta4 = z_d - L0
    
    # Check Z-axis limits
    if theta4 < JOINT_LIMITS[3, 0] or theta4 > JOINT_LIMITS[3, 1]:
        info['reachable'] = False
        info['message'] = f'Z out of range: θ₄={theta4:.4f} not in [{JOINT_LIMITS[3,0]:.2f}, {JOINT_LIMITS[3,1]:.2f}]'
        return None, info
    
    # Step 2: Solve for θ₂ (elbow angle)
    r_squared = x_d**2 + y_d**2
    r = np.sqrt(r_squared)
    
    # Check radial workspace
    if r > R_MAX:
        info['reachable'] = False
        info['message'] = f'Target too far: r={r:.4f} > R_max={R_MAX:.4f}'
        return None, info
    if r < R_MIN:
        info['reachable'] = False
        info['message'] = f'Target too close: r={r:.4f} < R_min={R_MIN:.4f}'
        return None, info
    
    # Law of cosines
    cos_theta2 = (r_squared - L1**2 - L2**2) / (2 * L1 * L2)
    
    # Clamp to valid range (handle numerical errors at workspace boundaries)
    cos_theta2 = np.clip(cos_theta2, -1.0, 1.0)
    
    # Two solutions: elbow right (+θ₂) and elbow left (-θ₂)
    if elbow_right:
        theta2 = np.arccos(cos_theta2)   # Positive angle (right)
    else:
        theta2 = -np.arccos(cos_theta2)  # Negative angle (left)
    
    # Check θ₂ joint limits
    if theta2 < JOINT_LIMITS[1, 0] or theta2 > JOINT_LIMITS[1, 1]:
        info['reachable'] = False
        info['message'] = f'θ₂={np.degrees(theta2):.1f}° out of range [{np.degrees(JOINT_LIMITS[1,0]):.0f}°, {np.degrees(JOINT_LIMITS[1,1]):.0f}°]'
        return None, info
    
    # Step 3: Solve for θ₁ (shoulder angle)
    beta = np.arctan2(y_d, x_d)
    psi = np.arctan2(L2 * np.sin(theta2), L1 + L2 * np.cos(theta2))
    theta1 = beta - psi
    
    # Normalize θ₁ to [-π, π]
    theta1 = np.arctan2(np.sin(theta1), np.cos(theta1))
    
    # Check θ₁ joint limits
    if theta1 < JOINT_LIMITS[0, 0] or theta1 > JOINT_LIMITS[0, 1]:
        info['reachable'] = False
        info['message'] = f'θ₁={np.degrees(theta1):.1f}° out of range [{np.degrees(JOINT_LIMITS[0,0]):.0f}°, {np.degrees(JOINT_LIMITS[0,1]):.0f}°]'
        return None, info
    
    # Step 4: Solve for θ₃ (wrist angle)
    theta3 = target_phi - theta1 - theta2
    
    # Normalize θ₃ to [-π, π]
    theta3 = np.arctan2(np.sin(theta3), np.cos(theta3))
    
    # Check θ₃ joint limits
    if theta3 < JOINT_LIMITS[2, 0] or theta3 > JOINT_LIMITS[2, 1]:
        info['reachable'] = False
        info['message'] = f'θ₃={np.degrees(theta3):.1f}° out of range [{np.degrees(JOINT_LIMITS[2,0]):.0f}°, {np.degrees(JOINT_LIMITS[2,1]):.0f}°]'
        return None, info
    
    return np.array([theta1, theta2, theta3, theta4]), info


print("Analytical IK function defined successfully!")


Analytical IK function defined successfully!


### Testing Analytical IK

Let's verify our IK solution by:
1. Computing FK for a known joint configuration
2. Using IK to recover the joint values from the resulting position
3. Checking that FK and IK are inverses of each other


In [5]:
# Test 1: Round-trip verification (FK → IK → FK)
print("=" * 60)
print("Test 1: Round-Trip Verification (FK → IK → FK)")
print("=" * 60)

# Original joint configuration
q_original = np.array([np.pi/6, np.pi/8, np.pi/4, 0.1])  # 30°, 22.5°, 45°, 100mm

# Forward kinematics
pos_original, phi_original = fk_scara(q_original)
print(f"\nOriginal joints: θ₁={np.degrees(q_original[0]):.1f}°, θ₂={np.degrees(q_original[1]):.1f}°, "
      f"θ₃={np.degrees(q_original[2]):.1f}°, θ₄={q_original[3]*1000:.1f}mm")
print(f"FK result: pos=({pos_original[0]:.4f}, {pos_original[1]:.4f}, {pos_original[2]:.4f}), φ={np.degrees(phi_original):.1f}°")

# Inverse kinematics
q_recovered, info = ik_scara_analytical(pos_original, phi_original, elbow_right=True)

if q_recovered is not None:
    # Verify with FK
    pos_check, phi_check = fk_scara(q_recovered)
    
    print(f"\nRecovered joints: θ₁={np.degrees(q_recovered[0]):.1f}°, θ₂={np.degrees(q_recovered[1]):.1f}°, "
          f"θ₃={np.degrees(q_recovered[2]):.1f}°, θ₄={q_recovered[3]*1000:.1f}mm")
    print(f"FK check: pos=({pos_check[0]:.4f}, {pos_check[1]:.4f}, {pos_check[2]:.4f}), φ={np.degrees(phi_check):.1f}°")
    
    # Calculate errors
    pos_error = np.linalg.norm(pos_original - pos_check)
    phi_error = abs(phi_original - phi_check)
    
    print(f"\n✓ Position error: {pos_error*1000:.6f} mm")
    print(f"✓ Orientation error: {np.degrees(phi_error):.6f}°")
else:
    print(f"\n✗ IK failed: {info['message']}")


Test 1: Round-Trip Verification (FK → IK → FK)

Original joints: θ₁=30.0°, θ₂=22.5°, θ₃=45.0°, θ₄=100.0mm
FK result: pos=(0.3816, 0.3087, 0.2000), φ=97.5°

Recovered joints: θ₁=30.0°, θ₂=22.5°, θ₃=45.0°, θ₄=100.0mm
FK check: pos=(0.3816, 0.3087, 0.2000), φ=97.5°

✓ Position error: 0.000000 mm
✓ Orientation error: 0.000000°


### Multiple Solutions: Elbow Left vs Elbow Right

For a **SCARA robot** (horizontal 2R arm), most reachable positions have **two** valid IK solutions:
- **Elbow Right** (+θ₂): Elbow bends to the right
- **Elbow Left** (-θ₂): Elbow bends to the left

This is a fundamental characteristic of IK - the same end-effector position can be achieved with different joint angles.


In [6]:
# Test 2: Multiple solutions (elbow left vs elbow right)
print("=" * 60)
print("Test 2: Multiple Solutions (Elbow Left vs Elbow Right)")
print("=" * 60)

# Target position in the middle of workspace
target = np.array([0.35, 0.15, 0.70])
target_phi = 0.0

print(f"\nTarget position: ({target[0]:.3f}, {target[1]:.3f}, {target[2]:.3f}) m")
print(f"Target orientation: {np.degrees(target_phi):.1f}°")

# Solve for both configurations
q_right, info_right = ik_scara_analytical(target, target_phi, elbow_right=True)
q_left, info_left = ik_scara_analytical(target, target_phi, elbow_right=False)

print("\n--- Elbow RIGHT Configuration (+θ₂) ---")
if q_right is not None:
    pos_right, phi_right = fk_scara(q_right)
    print(f"  θ₁ = {np.degrees(q_right[0]):7.2f}°")
    print(f"  θ₂ = {np.degrees(q_right[1]):7.2f}° (positive)")
    print(f"  θ₃ = {np.degrees(q_right[2]):7.2f}°")
    print(f"  θ₄ = {q_right[3]*1000:7.2f} mm")
    print(f"  FK verify: ({pos_right[0]:.4f}, {pos_right[1]:.4f}, {pos_right[2]:.4f})")
else:
    print(f"  Not reachable: {info_right['message']}")

print("\n--- Elbow LEFT Configuration (-θ₂) ---")
if q_left is not None:
    pos_left, phi_left = fk_scara(q_left)
    print(f"  θ₁ = {np.degrees(q_left[0]):7.2f}°")
    print(f"  θ₂ = {np.degrees(q_left[1]):7.2f}° (negative)")
    print(f"  θ₃ = {np.degrees(q_left[2]):7.2f}°")
    print(f"  θ₄ = {q_left[3]*1000:7.2f} mm")
    print(f"  FK verify: ({pos_left[0]:.4f}, {pos_left[1]:.4f}, {pos_left[2]:.4f})")
else:
    print(f"  Not reachable: {info_left['message']}")


Test 2: Multiple Solutions (Elbow Left vs Elbow Right)

Target position: (0.350, 0.150, 0.700) m
Target orientation: 0.0°

--- Elbow RIGHT Configuration (+θ₂) ---
  Not reachable: Z out of range: θ₄=0.6000 not in [0.00, 0.25]

--- Elbow LEFT Configuration (-θ₂) ---
  Not reachable: Z out of range: θ₄=0.6000 not in [0.00, 0.25]


### Visualizing the Workspace

The **workspace** is the set of all positions the end-effector can reach. For our SCARA robot:
- **Horizontal workspace**: An annular region (ring) with inner radius $|\ell_1 - \ell_2|$ and outer radius $\ell_1 + \ell_2$
- **Vertical workspace**: Range determined by the prismatic joint stroke

Let's visualize the reachable workspace and test various target positions.


In [7]:
# ============================================================
# 3D Workspace Visualization - Using Newton's IK for Accuracy
# ============================================================
# Newton's IK respects the actual MJCF joint limits, giving accurate results
import rerun as rr

# Create viewer
workspace_viewer = newton.viewer.ViewerRerun()

# Generate a uniform grid in Cartesian space (X, Y, Z)
print("Generating workspace grid using Newton's IK...")
print("  (Newton uses actual MJCF joint limits: θ₁=±90°, θ₂=±45°)")

# Grid parameters
x_range = np.linspace(-0.1, 0.6, 25)
y_range = np.linspace(-0.4, 0.4, 25)
z_range = np.linspace(0.55, 0.90, 8)

# Create a Newton IK solver for reachability testing
test_target = wp.array([wp.vec3(0.0, 0.0, 0.0)], dtype=wp.vec3)
test_pos_obj = newton.ik.IKPositionObjective(
    link_index=ee_body_idx,
    link_offset=wp.vec3(0.0, 0.0, 0.0),
    target_positions=test_target,
    weight=1.0,
)
test_limit_obj = newton.ik.IKJointLimitObjective(
    joint_limit_lower=model.joint_limit_lower,
    joint_limit_upper=model.joint_limit_upper,
    weight=10.0,
)

test_solver = newton.ik.IKSolver(
    model=model,
    n_problems=1,
    objectives=[test_pos_obj, test_limit_obj],
    jacobian_mode=newton.ik.IKJacobianMode.ANALYTIC,
    optimizer=newton.ik.IKOptimizer.LM,
    lambda_initial=0.01,
)

def is_reachable_newton(target_pos, threshold_mm=5.0):
    """Test if a position is reachable using Newton's IK solver."""
    # Set target
    test_pos_obj.set_target_position(0, wp.vec3(target_pos[0], target_pos[1], target_pos[2]))
    
    # Reset to home position
    joint_q = wp.array(initial_q.reshape(1, -1), dtype=wp.float32)
    
    # Solve
    test_solver.step(joint_q, joint_q, iterations=50, step_size=1.0)
    
    # Check result via FK
    q_sol = joint_q.numpy()[0]
    joint_q_np = state.joint_q.numpy()
    for i in range(min(len(q_sol), len(joint_q_np))):
        joint_q_np[i] = q_sol[i]
    state.joint_q.assign(joint_q_np)
    newton.eval_fk(model, state.joint_q, state.joint_qd, state)
    body_q_np = state.body_q.numpy()
    achieved_pos = body_q_np[ee_body_idx][:3]
    
    error_mm = np.linalg.norm(target_pos - achieved_pos) * 1000
    return error_mm < threshold_mm

reachable_points = []
reachable_colors = []
unreachable_points = []

total_points = len(x_range) * len(y_range) * len(z_range)
checked = 0

for x in x_range:
    for y in y_range:
        for z in z_range:
            target = np.array([x, y, z], dtype=np.float32)
            
            # Use Newton's IK to test reachability
            if is_reachable_newton(target, threshold_mm=5.0):
                reachable_points.append([x, y, z])
                # Color by Z height
                z_norm = (z - z_range[0]) / (z_range[-1] - z_range[0])
                if z_norm < 0.5:
                    t = z_norm * 2
                    reachable_colors.append([int(50*(1-t)), int(200*t + 100), int(255*(1-t))])
                else:
                    t = (z_norm - 0.5) * 2
                    reachable_colors.append([int(200*t + 50), int(255 - 55*t), int(50*(1-t))])
            else:
                unreachable_points.append([x, y, z])
            
            checked += 1
            if checked % 500 == 0:
                print(f"  Progress: {checked}/{total_points} points checked...")

reachable_points = np.array(reachable_points) if reachable_points else np.zeros((0, 3))
unreachable_points = np.array(unreachable_points) if unreachable_points else np.zeros((0, 3))

print(f"\n  Grid: {len(x_range)}x{len(y_range)}x{len(z_range)} = {total_points} points")
print(f"  Reachable (Newton IK): {len(reachable_points)} points")
print(f"  Unreachable: {len(unreachable_points)} points")

# Log reachable points
if len(reachable_points) > 0:
    rr.log("workspace/reachable", 
           rr.Points3D(reachable_points, colors=reachable_colors, radii=0.004))

# Log unreachable points
if len(unreachable_points) > 0:
    rr.log("workspace/unreachable", 
           rr.Points3D(unreachable_points, colors=[[255, 50, 50, 80]], radii=0.003))

print(f"\n✓ 3D workspace (Newton IK) ready!")
print(f"  - Uses actual MJCF joint limits")
print(f"  - Colored spheres: reachable")
print(f"  - Red spheres: unreachable")
workspace_viewer


Generating workspace grid using Newton's IK...
  (Newton uses actual MJCF joint limits: θ₁=±90°, θ₂=±45°)


NameError: name 'ee_body_idx' is not defined

## Part 2: Numerical Inverse Kinematics

### The Jacobian Method

While analytical IK works well for simple robots like SCARA, most industrial robots require **numerical methods**. The standard approach uses the **Jacobian matrix**.

### The Jacobian Matrix

The Jacobian $J$ relates joint velocities to end-effector velocities:

$$\dot{x} = J(q) \dot{q}$$

Where:
- $\dot{x}$ is the end-effector velocity (linear + angular)
- $\dot{q}$ is the joint velocity vector
- $J(q)$ is the 6×n Jacobian matrix (depends on current configuration)

### Jacobian for Position-Only IK

For our SCARA robot (position only), the 3×4 Jacobian is:

$$J = \begin{bmatrix}
\frac{\partial x}{\partial \theta_1} & \frac{\partial x}{\partial \theta_2} & \frac{\partial x}{\partial \theta_3} & \frac{\partial x}{\partial \theta_4} \\
\frac{\partial y}{\partial \theta_1} & \frac{\partial y}{\partial \theta_2} & \frac{\partial y}{\partial \theta_3} & \frac{\partial y}{\partial \theta_4} \\
\frac{\partial z}{\partial \theta_1} & \frac{\partial z}{\partial \theta_2} & \frac{\partial z}{\partial \theta_3} & \frac{\partial z}{\partial \theta_4}
\end{bmatrix}$$

Computing the partial derivatives:

$$J = \begin{bmatrix}
-\ell_1 s_1 - \ell_2 s_{12} & -\ell_2 s_{12} & 0 & 0 \\
\ell_1 c_1 + \ell_2 c_{12} & \ell_2 c_{12} & 0 & 0 \\
0 & 0 & 0 & 1
\end{bmatrix}$$

Where $s_1 = \sin\theta_1$, $c_1 = \cos\theta_1$, $s_{12} = \sin(\theta_1+\theta_2)$, $c_{12} = \cos(\theta_1+\theta_2)$

### Numerical IK Algorithm

The **Jacobian Pseudo-inverse** method iteratively updates joint angles:

$$q_{k+1} = q_k + J^{\dagger}(x_d - x_k)$$

Where $J^{\dagger} = J^T(JJ^T)^{-1}$ is the pseudo-inverse.

For better stability, we use **Damped Least Squares (Levenberg-Marquardt)**:

$$q_{k+1} = q_k + J^T(JJ^T + \lambda^2 I)^{-1}(x_d - x_k)$$

The damping parameter $\lambda$ provides stability near singularities.


### Implementation of Jacobian and Numerical IK

Below we implement:
1. `jacobian_scara(q)`: Computes the 3×4 position Jacobian
2. `ik_scara_numerical(target, q_init)`: Iterative IK using damped least squares


In [ ]:
def jacobian_scara(q):
    """
    Compute the 3×4 position Jacobian for the SCARA robot.
    
    J = [∂x/∂θ₁, ∂x/∂θ₂, ∂x/∂θ₃, ∂x/∂θ₄]
        [∂y/∂θ₁, ∂y/∂θ₂, ∂y/∂θ₃, ∂y/∂θ₄]
        [∂z/∂θ₁, ∂z/∂θ₂, ∂z/∂θ₃, ∂z/∂θ₄]
    
    Parameters:
        q: array [θ₁, θ₂, θ₃, θ₄]
    
    Returns:
        J: 3×4 Jacobian matrix
    """
    theta1, theta2, theta3, theta4 = q
    
    s1 = np.sin(theta1)
    c1 = np.cos(theta1)
    s12 = np.sin(theta1 + theta2)
    c12 = np.cos(theta1 + theta2)
    
    # Jacobian matrix
    J = np.array([
        [-L1*s1 - L2*s12,  -L2*s12,  0,  0],  # ∂x/∂θᵢ
        [ L1*c1 + L2*c12,   L2*c12,  0,  0],  # ∂y/∂θᵢ
        [              0,        0,  0,  1]   # ∂z/∂θᵢ
    ])
    
    return J


def ik_scara_numerical(target_pos, q_init, max_iter=100, tol=1e-6, damping=0.1):
    """
    Numerical IK using damped least squares (Levenberg-Marquardt).
    
    Parameters:
        target_pos: array [x, y, z] - desired position
        q_init: array [θ₁, θ₂, θ₃, θ₄] - initial guess
        max_iter: maximum iterations
        tol: convergence tolerance (m)
        damping: Levenberg-Marquardt damping factor
    
    Returns:
        q: final joint configuration
        info: dict with convergence information
    """
    q = q_init.copy()
    
    info = {
        'converged': False,
        'iterations': 0,
        'final_error': float('inf'),
        'error_history': []
    }
    
    for i in range(max_iter):
        # Current position (FK)
        pos_current, _ = fk_scara(q)
        
        # Position error
        error = target_pos - pos_current
        error_norm = np.linalg.norm(error)
        info['error_history'].append(error_norm)
        
        # Check convergence
        if error_norm < tol:
            info['converged'] = True
            info['iterations'] = i + 1
            info['final_error'] = error_norm
            break
        
        # Compute Jacobian
        J = jacobian_scara(q)
        
        # Damped least squares: Δq = J^T (J J^T + λ²I)^(-1) error
        JJT = J @ J.T
        damped = JJT + damping**2 * np.eye(3)
        delta_q = J.T @ np.linalg.solve(damped, error)
        
        # Update joint angles
        q = q + delta_q
        
        # Apply joint limits
        q = np.clip(q, JOINT_LIMITS[:, 0], JOINT_LIMITS[:, 1])
    
    info['iterations'] = max_iter
    info['final_error'] = np.linalg.norm(target_pos - fk_scara(q)[0])
    
    return q, info


print("Jacobian and numerical IK functions defined!")


### Limitations of Manual Numerical IK

Our simple damped least-squares implementation works but has limitations:
- May not converge for difficult targets (near joint limits or singularities)
- No GPU acceleration
- Single problem at a time

Let's see this in action, then learn how Newton solves these issues.


In [ ]:
# Test numerical IK
print("=" * 60)
print("Testing Numerical IK (Damped Least Squares)")
print("=" * 60)

# Target position
target = np.array([0.35, 0.15, 0.70])
print(f"\nTarget: ({target[0]:.3f}, {target[1]:.3f}, {target[2]:.3f})")

# Initial guess (home position)
q_init = np.array([0.0, 0.0, 0.0, 0.1])
print(f"Initial guess: θ₁={np.degrees(q_init[0]):.1f}°, θ₂={np.degrees(q_init[1]):.1f}°, "
      f"θ₃={np.degrees(q_init[2]):.1f}°, θ₄={q_init[3]*1000:.1f}mm")

# Solve numerically
q_numerical, info = ik_scara_numerical(target, q_init)
pos_numerical, _ = fk_scara(q_numerical)

print(f"\n--- Numerical IK Result ---")
print(f"  Converged: {info['converged']}")
print(f"  Iterations: {info['iterations']}")
print(f"  Final error: {info['final_error']*1000:.6f} mm")
print(f"\n  θ₁ = {np.degrees(q_numerical[0]):7.2f}°")
print(f"  θ₂ = {np.degrees(q_numerical[1]):7.2f}°")
print(f"  θ₃ = {np.degrees(q_numerical[2]):7.2f}°")
print(f"  θ₄ = {q_numerical[3]*1000:7.2f} mm")
print(f"\n  FK verify: ({pos_numerical[0]:.4f}, {pos_numerical[1]:.4f}, {pos_numerical[2]:.4f})")

# Compare with analytical
q_analytical, _ = ik_scara_analytical(target, 0.0, elbow_right=True)
if q_analytical is not None:
    print(f"\n--- Analytical IK (for comparison) ---")
    print(f"  θ₁ = {np.degrees(q_analytical[0]):7.2f}°")
    print(f"  θ₂ = {np.degrees(q_analytical[1]):7.2f}°")
    print(f"  θ₃ = {np.degrees(q_analytical[2]):7.2f}°")
    print(f"  θ₄ = {q_analytical[3]*1000:7.2f} mm")


The manual solver struggles with this target (hitting joint limits). Now let's see how Newton's built-in IK solver handles this much better.


## Part 3: Newton's Production IK Solver

Newton provides `newton.ik` - a **production-ready IK solver** that overcomes the limitations of manual implementations:

| Feature | Manual Implementation | Newton IK |
|---------|----------------------|-----------|
| **Speed** | Pure Python/NumPy | GPU-accelerated Warp kernels |
| **Robustness** | May fail near limits | Adaptive damping handles edge cases |
| **Parallelism** | One problem at a time | Batch solve 1000s simultaneously |
| **Flexibility** | Custom code needed | Composable objectives |

### Key Components

```python
newton.ik.IKSolver           # Main solver class
newton.ik.IKPositionObjective    # End-effector position target
newton.ik.IKRotationObjective    # End-effector orientation target  
newton.ik.IKJointLimitObjective  # Keep joints in bounds
```


In [ ]:
# First, load the SCARA robot model from MJCF
builder = newton.ModelBuilder()
mjcf_path = os.path.join(os.getcwd(), "scara_rrrp.xml")
builder.add_mjcf(mjcf_path)
model = builder.finalize()
state = model.state()

# Set initial joint configuration
initial_q = np.array([0.0, 0.0, 0.0, 0.1], dtype=np.float32)
joint_q_np = state.joint_q.numpy()
for i, q_val in enumerate(initial_q):
    if i < len(joint_q_np):
        joint_q_np[i] = q_val
state.joint_q.assign(joint_q_np)

print(f"Model loaded: {mjcf_path}")
print(f"  Bodies: {model.body_count}, Joints: {model.joint_count}")
print(f"  Joint coordinates: {model.joint_coord_count}")

# Find the end-effector body index
# For SCARA, use the "ee" body (end-effector site), not "piston"
print("\nAvailable bodies:")
for i, key in enumerate(model.body_key):
    print(f"  {i}: {key}")

# Use the last body (ee) as end-effector
ee_body_idx = model.body_count - 1
print(f"\nUsing end-effector body: {ee_body_idx} ({model.body_key[ee_body_idx]})")

# Get initial end-effector pose via FK
newton.eval_fk(model, state.joint_q, state.joint_qd, state)
body_q_np = state.body_q.numpy()
ee_initial_tf = wp.transform(*body_q_np[ee_body_idx])
ee_initial_pos = wp.transform_get_translation(ee_initial_tf)
ee_initial_rot = wp.transform_get_rotation(ee_initial_tf)

print(f"Initial EE position: ({ee_initial_pos[0]:.4f}, {ee_initial_pos[1]:.4f}, {ee_initial_pos[2]:.4f})")


### Understanding Residuals

The IK solver works by minimizing **residuals** - error terms that measure how far we are from our goals:

$$\text{minimize} \quad ||r||^2 + \lambda ||\Delta q||^2$$

Where $r$ is the residual vector and $\lambda$ is the damping factor for stability.

### Residual Layout

Each objective contributes residuals to a combined vector. For a typical IK problem:

```
residuals = [
    r_position[0:3],      # Position error (x, y, z) 
    r_rotation[3:6],      # Rotation error (axis-angle)
    r_limits[6:6+n_dofs]  # Joint limit violations
]
```

### IK Objectives

| Objective | Description | Residuals | When to Use |
|-----------|-------------|-----------|-------------|
| `IKPositionObjective` | End-effector position | 3 (x,y,z) | Always - reach target position |
| `IKRotationObjective` | End-effector orientation | 3 (axis-angle) | Grasping, tool use - control gripper angle |
| `IKJointLimitObjective` | Keep joints in bounds | n_dofs | Recommended - prevents invalid configs |


### Solving IK with Newton's Solver

Now let's test Newton's IK solver on the same target position and compare with our manual implementation.


In [ ]:
# Test Newton's IK solver with BOTH position AND rotation objectives
print("=" * 60)
print("Testing Newton's IK Solver (Position + Rotation)")
print("=" * 60)

# Reset to initial config and get EE pose
joint_q_np = state.joint_q.numpy()
for i, q in enumerate(initial_q):
    if i < len(joint_q_np):
        joint_q_np[i] = q
state.joint_q.assign(joint_q_np)
newton.eval_fk(model, state.joint_q, state.joint_qd, state)
body_q_np = state.body_q.numpy()

# Get initial EE position and rotation
newton_initial_pos = body_q_np[ee_body_idx][:3].copy()
newton_initial_rot = body_q_np[ee_body_idx][3:7].copy()  # quaternion (x,y,z,w)

print(f"\nInitial EE position: ({newton_initial_pos[0]:.4f}, {newton_initial_pos[1]:.4f}, {newton_initial_pos[2]:.4f})")
print(f"Initial EE rotation (quat): ({newton_initial_rot[0]:.3f}, {newton_initial_rot[1]:.3f}, {newton_initial_rot[2]:.3f}, {newton_initial_rot[3]:.3f})")

# Define target: move in X-Y plane, MAINTAIN orientation
target_offset = np.array([-0.02, 0.02, 0.0], dtype=np.float32)
target_pos = newton_initial_pos + target_offset
target_rot = newton_initial_rot  # Keep same orientation

print(f"\nTarget position: ({target_pos[0]:.4f}, {target_pos[1]:.4f}, {target_pos[2]:.4f})")
print(f"Target rotation: maintain initial orientation")

# Create objectives
# 1. Position objective (weight=1.0)
test_pos_obj = newton.ik.IKPositionObjective(
    link_index=ee_body_idx,
    link_offset=wp.vec3(0.0, 0.0, 0.0),
    target_positions=wp.array([target_pos], dtype=wp.vec3),
    weight=1.0,
)

# 2. Rotation objective (weight=0.5) - maintain end-effector orientation
test_rot_obj = newton.ik.IKRotationObjective(
    link_index=ee_body_idx,
    link_offset_rotation=wp.quat_identity(),
    target_rotations=wp.array([wp.vec4(target_rot[0], target_rot[1], target_rot[2], target_rot[3])], dtype=wp.vec4),
    weight=0.5,  # Lower weight - position is primary goal
)

# 3. Joint limit objective (weight=10.0)
test_limit_obj = newton.ik.IKJointLimitObjective(
    joint_limit_lower=model.joint_limit_lower,
    joint_limit_upper=model.joint_limit_upper,
    weight=10.0,
)

# Create solver with all three objectives
test_solver = newton.ik.IKSolver(
    model=model,
    n_problems=1,
    objectives=[test_pos_obj, test_rot_obj, test_limit_obj],  # All three!
    jacobian_mode=newton.ik.IKJacobianMode.ANALYTIC,
    lambda_initial=0.01,
)

# Solve IK
joint_q_2d = wp.array(initial_q, dtype=wp.float32, shape=(1, model.joint_coord_count))
ik_iterations = 100
test_solver.step(joint_q_2d, joint_q_2d, iterations=ik_iterations, step_size=1.0)

# Get result and verify
q_newton = joint_q_2d.numpy()[0]
joint_q_np = state.joint_q.numpy()
for i in range(min(len(q_newton), len(joint_q_np))):
    joint_q_np[i] = q_newton[i]
state.joint_q.assign(joint_q_np)
newton.eval_fk(model, state.joint_q, state.joint_qd, state)
body_q_np = state.body_q.numpy()

ee_final_pos = body_q_np[ee_body_idx][:3]
ee_final_rot = body_q_np[ee_body_idx][3:7]

# Calculate errors
position_error = np.linalg.norm(target_pos - ee_final_pos)
# Rotation error: angle between quaternions
dot_product = np.abs(np.dot(target_rot, ee_final_rot))
rotation_error_rad = 2 * np.arccos(np.clip(dot_product, -1, 1))

print(f"\n--- Newton IK Result (with Rotation) ---")
print(f"  Iterations: {ik_iterations}")
print(f"  Position error: {position_error*1000:.4f} mm")
print(f"  Rotation error: {np.degrees(rotation_error_rad):.4f}°")
print(f"\n  θ₁ = {np.degrees(q_newton[0]):7.2f}°")
print(f"  θ₂ = {np.degrees(q_newton[1]):7.2f}°")
print(f"  θ₃ = {np.degrees(q_newton[2]):7.2f}°  ← controls EE orientation")
print(f"  θ₄ = {q_newton[3]*1000:7.2f} mm")

if position_error < 0.001 and rotation_error_rad < 0.01:
    print("\n✓ Newton IK converged (position + rotation)!")
elif position_error < 0.001:
    print("\n✓ Position converged, rotation within tolerance")
else:
    print(f"\n⚠ Position error: {position_error*1000:.2f}mm")


### Newton IK Configuration Options

Newton's IK solver is highly configurable:

**Jacobian Modes:**
| Mode | Description | When to Use |
|------|-------------|-------------|
| `ANALYTIC` | Hand-derived Jacobians via Warp kernels | Standard position/rotation IK (fastest) |
| `AUTODIFF` | Automatic differentiation | Custom objectives, prototyping |

**Optimizers:**
| Optimizer | Algorithm | When to Use |
|-----------|-----------|-------------|
| `LM` | Levenberg-Marquardt | Most IK problems, handles singularities well |
| `LBFGS` | Limited-memory BFGS | Large DOF systems (humanoids, snakes) |

For our SCARA robot, we use **ANALYTIC + LM** for best performance.


## Part 4: IK for Trajectories

Newton supports two approaches for trajectory IK:

### Approach A: Batched IK (Offline Planning)
Solve IK for **all waypoints at once** on the GPU - great for trajectory planning.

### Approach B: Real-Time IK (Online Control)  
Update targets dynamically with `set_target_position()` - essential for teleoperation.

Let's demonstrate both approaches!


In [ ]:
# Batched IK: Solve IK for entire circular trajectory at once
print("=" * 60)
print("Batched IK: Solving Trajectory in Parallel")
print("=" * 60)

# Get Newton's actual EE position to use correct coordinate frame
joint_q_np = state.joint_q.numpy()
for i, q in enumerate(initial_q):
    if i < len(joint_q_np):
        joint_q_np[i] = q
state.joint_q.assign(joint_q_np)
newton.eval_fk(model, state.joint_q, state.joint_qd, state)
body_q_np = state.body_q.numpy()
newton_ee_pos = body_q_np[ee_body_idx][:3].copy()

print(f"Newton EE position: ({newton_ee_pos[0]:.3f}, {newton_ee_pos[1]:.3f}, {newton_ee_pos[2]:.3f})")

# Generate trajectory waypoints (small circle in X-Y plane)
n_waypoints = 50
circle_radius = 0.03  # 3cm radius circle
# Use Newton's coordinate frame - center the circle slightly offset from current position
center = np.array([newton_ee_pos[0] - 0.02, newton_ee_pos[1], newton_ee_pos[2]], dtype=np.float32)
angles = np.linspace(0, 2*np.pi, n_waypoints, endpoint=False)

# Create target positions for all waypoints
trajectory_targets = []
for angle in angles:
    x = center[0] + circle_radius * np.cos(angle)
    y = center[1] + circle_radius * np.sin(angle)
    z = center[2]
    trajectory_targets.append([x, y, z])
trajectory_targets = np.array(trajectory_targets, dtype=np.float32)

print(f"\nTrajectory: {n_waypoints} waypoints")
print(f"Center: ({center[0]:.3f}, {center[1]:.3f}, {center[2]:.3f})")
print(f"Radius: {circle_radius*1000:.0f} mm")


In [ ]:
# Create batched IK solver for all waypoints
import time

# Create objectives for batched solving
batched_pos_obj = newton.ik.IKPositionObjective(
    link_index=ee_body_idx,
    link_offset=wp.vec3(0.0, 0.0, 0.0),
    target_positions=wp.array(trajectory_targets, dtype=wp.vec3),
    weight=1.0,
)

batched_joint_limit_obj = newton.ik.IKJointLimitObjective(
    joint_limit_lower=model.joint_limit_lower,
    joint_limit_upper=model.joint_limit_upper,
    weight=10.0,
)

# Create batched solver
batched_solver = newton.ik.IKSolver(
    model=model,
    n_problems=n_waypoints,  # Solve all waypoints in parallel!
    objectives=[batched_pos_obj, batched_joint_limit_obj],
    optimizer=newton.ik.IKOptimizer.LM,
    jacobian_mode=newton.ik.IKJacobianMode.ANALYTIC,
    lambda_initial=0.1,
)

# Initialize joint configurations for all problems
# Start from the same initial configuration
initial_q_batched = np.tile(initial_q, (n_waypoints, 1)).astype(np.float32)
joint_q_batched = wp.array(initial_q_batched, shape=(n_waypoints, model.joint_coord_count))

print(f"\nBatched IK Solver created:")
print(f"  Problems in parallel: {n_waypoints}")
print(f"  Joint array shape: {joint_q_batched.shape}")


In [ ]:
# Solve all IK problems in parallel!
print("\nSolving all waypoints in parallel...")

wp.synchronize()
t0 = time.perf_counter()
batched_solver.step(joint_q_batched, joint_q_batched, iterations=100, step_size=1.0)
wp.synchronize()
solve_time = (time.perf_counter() - t0) * 1000

print(f"✓ Solved {n_waypoints} IK problems in {solve_time:.2f} ms")
print(f"  Average per problem: {solve_time/n_waypoints:.3f} ms")

# Get results and verify using Newton's FK (not manual FK!)
joint_solutions = joint_q_batched.numpy()

# Compute achieved positions using Newton's FK
achieved_positions = []
for i in range(n_waypoints):
    q_sol = joint_solutions[i]
    # Use Newton's FK for consistency
    joint_q_np = state.joint_q.numpy()
    for j in range(min(len(q_sol), len(joint_q_np))):
        joint_q_np[j] = q_sol[j]
    state.joint_q.assign(joint_q_np)
    newton.eval_fk(model, state.joint_q, state.joint_qd, state)
    body_q_np = state.body_q.numpy()
    pos = body_q_np[ee_body_idx][:3].copy()
    achieved_positions.append(pos)
achieved_positions = np.array(achieved_positions)

# Compute errors
errors = np.linalg.norm(trajectory_targets - achieved_positions, axis=1)
print(f"\nTracking accuracy:")
print(f"  Mean error: {np.mean(errors)*1000:.4f} mm")
print(f"  Max error:  {np.max(errors)*1000:.4f} mm")
print(f"  Min error:  {np.min(errors)*1000:.4f} mm")


In [ ]:
# Visualize batched IK results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Target vs Achieved trajectory (top view)
ax1 = axes[0]
ax1.plot(trajectory_targets[:, 0], trajectory_targets[:, 1], 'b-', linewidth=2, label='Target', alpha=0.7)
ax1.plot(achieved_positions[:, 0], achieved_positions[:, 1], 'r--', linewidth=2, label='Achieved', alpha=0.7)
ax1.scatter(trajectory_targets[0, 0], trajectory_targets[0, 1], c='green', s=100, marker='o', label='Start', zorder=5)
ax1.set_xlabel('X (m)')
ax1.set_ylabel('Y (m)')
ax1.set_title('Batched IK Results - Top View', fontweight='bold')
ax1.legend()
ax1.set_aspect('equal')
ax1.grid(True, alpha=0.3)

# Plot 2: Position error over trajectory
ax2 = axes[1]
ax2.plot(errors * 1000, 'g-', linewidth=2)
ax2.axhline(y=np.mean(errors)*1000, color='r', linestyle='--', label=f'Mean: {np.mean(errors)*1000:.3f} mm')
ax2.fill_between(range(len(errors)), 0, errors*1000, alpha=0.3, color='green')
ax2.set_xlabel('Waypoint Index')
ax2.set_ylabel('Position Error (mm)')
ax2.set_title('IK Tracking Error', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('batched_ik_results.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Batched IK visualization saved as 'batched_ik_results.png'")


In [ ]:
# ============================================================
# Approach B: Real-Time IK with set_target_position()
# ============================================================
# For real-time control, update targets dynamically in the loop
# Key difference: we call set_target_position() then solver.step()

# Create a solver for real-time use (reuse position objective)
realtime_pos_obj = newton.ik.IKPositionObjective(
    link_index=ee_body_idx,
    link_offset=wp.vec3(0.0, 0.0, 0.0),
    target_positions=wp.array([trajectory_targets[0]], dtype=wp.vec3),  # Start position
    weight=1.0,
)

realtime_solver = newton.ik.IKSolver(
    model=model,
    n_problems=1,  # Single problem for real-time
    objectives=[realtime_pos_obj],
    jacobian_mode=newton.ik.IKJacobianMode.ANALYTIC,
    lambda_initial=0.1,
)

# Reset joint config
realtime_joint_q = wp.array(initial_q, dtype=wp.float32, shape=(1, model.joint_coord_count))

# Create viewer
viewer = newton.viewer.ViewerRerun(keep_historical_data=True)
viewer.set_model(model)

sim_time = 0.0
frame_dt = 1.0 / 30.0
ik_iters_per_frame = 10  # Fewer iterations for real-time (faster)

print("Real-time IK trajectory following...")
print(f"  IK iterations per frame: {ik_iters_per_frame}")

for i in trange(n_waypoints, desc="Real-time IK"):
    # KEY: Update target position dynamically
    target = trajectory_targets[i]
    realtime_pos_obj.set_target_position(0, wp.vec3(target[0], target[1], target[2]))
    
    # Solve IK (few iterations for real-time responsiveness)
    realtime_solver.step(realtime_joint_q, realtime_joint_q, iterations=ik_iters_per_frame, step_size=1.0)
    
    # Apply solution
    q_sol = realtime_joint_q.numpy()[0]
    joint_q_np = state.joint_q.numpy()
    for j in range(min(len(q_sol), len(joint_q_np))):
        joint_q_np[j] = q_sol[j]
    state.joint_q.assign(joint_q_np)
    newton.eval_fk(model, state.joint_q, state.joint_qd, state)
    
    # Visualize
    viewer.begin_frame(sim_time)
    viewer.log_state(state)
    viewer.end_frame()
    sim_time += frame_dt

print(f"\n✓ Real-time IK complete! {n_waypoints} frames")
print("  Note: set_target_position() allows dynamic target updates")
viewer


## Part 5: Understanding Singularities (with Newton)

### What is a Singularity?

A **singularity** occurs when the Jacobian matrix loses rank - the robot loses a degree of freedom in Cartesian space:

- Cannot move in certain directions
- Would require infinite joint velocities
- Numerical IK becomes unstable

### Newton's Jacobian Computation

Newton's IKSolver computes Jacobians internally using two modes:

| Mode | Method | Speed | Use Case |
|------|--------|-------|----------|
| `ANALYTIC` | Motion subspace multiplication | Fast | Production, GPU batched |
| `AUTODIFF` | Warp reverse-mode autodiff | Flexible | Custom objectives |

The Jacobian is stored in `solver._impl.jacobian` with shape `(n_batch, n_residuals, n_dofs)`.

### Measuring Manipulability

The **manipulability measure** $\sqrt{\det(JJ^T)}$ indicates how well the robot can move. Near singularities, this value approaches zero.


In [ ]:
# ============================================================
# Using Newton's IKSolver to Extract Jacobians
# ============================================================
# Newton computes Jacobians internally during IK - we can access them!

def get_jacobian_from_newton_ik(model, body_idx, q_values, mode="ANALYTIC"):
    """
    Extract the Jacobian matrix using Newton's IK solver.
    
    This uses Newton's optimized Jacobian computation (ANALYTIC or AUTODIFF)
    rather than rolling our own autodiff code.
    """
    # Create a minimal IK solver just for Jacobian computation
    target_pos = wp.array([wp.vec3(0.0, 0.0, 0.0)], dtype=wp.vec3)
    pos_obj = newton.ik.IKPositionObjective(
        link_index=body_idx,
        link_offset=wp.vec3(0.0, 0.0, 0.0),
        target_positions=target_pos,
        weight=1.0,
    )
    
    jac_mode = newton.ik.IKJacobianMode.ANALYTIC if mode == "ANALYTIC" else newton.ik.IKJacobianMode.AUTODIFF
    
    solver = newton.ik.IKSolver(
        model=model,
        n_problems=1,
        objectives=[pos_obj],
        jacobian_mode=jac_mode,
        optimizer=newton.ik.IKOptimizer.LM,
    )
    
    # Set joint configuration
    joint_q = wp.array(q_values.astype(np.float32).reshape(1, -1), dtype=wp.float32)
    
    # Run one step to compute Jacobian (the solver computes J internally)
    solver.step(joint_q, joint_q, iterations=1)
    
    # Extract the Jacobian from the solver's internal buffer
    # Shape: (n_batch=1, n_residuals=3, n_dofs=4)
    J_newton = solver._impl.jacobian.numpy()[0]  # Remove batch dimension
    
    return J_newton  # Shape: (3, n_dofs)

# ============================================================
# Singularity Analysis using Newton's Jacobian
# ============================================================
print("=" * 60)
print("Singularity Analysis (using Newton's IKSolver Jacobian)")
print("=" * 60)

test_configs = [
    ([0.0, 0.0, 0.0, 0.1], "Fully Extended (singular)"),
    ([0.0, np.pi/4, 0.0, 0.1], "Normal Configuration"),
    ([0.0, np.pi/2, 0.0, 0.1], "Elbow at 90°"),
    ([np.pi/4, np.pi/4, 0.0, 0.1], "Rotated"),
]

print(f"\n{'Configuration':<30} | {'det(JJᵀ)':<15} | {'Rank':<6} | {'Status'}")
print("-" * 75)

for q, name in test_configs:
    q = np.array(q, dtype=np.float32)
    
    # Get Jacobian from Newton's IK solver (ANALYTIC mode)
    J = get_jacobian_from_newton_ik(model, ee_body_idx, q, mode="ANALYTIC")
    
    # Compute manipulability metrics
    JJT = J @ J.T
    det_val = np.linalg.det(JJT)
    rank = np.linalg.matrix_rank(J)
    
    if det_val < 1e-6:
        status = "⚠ SINGULAR"
    elif det_val < 1e-3:
        status = "Near singular"
    else:
        status = "✓ Normal"
    
    print(f"{name:<30} | {det_val:<15.6f} | {rank:<6} | {status}")

print("-" * 75)
print("\n✓ Jacobian computed via Newton's IKSolver (ANALYTIC mode)")
print("  → Uses motion subspace for efficient GPU computation")


In [ ]:
# ============================================================
# Manipulability Analysis
# ============================================================
# For efficiency, we use the analytical Jacobian formula for the sweep.
# Newton's IKSolver (ANALYTIC mode) computes the same Jacobian internally
# using the motion subspace - both approaches give identical results.

theta2_range = np.linspace(-np.pi/2, np.pi/2, 100)
manipulability = []

# Compare Newton vs Analytical at one point
q_test = np.array([0.0, np.pi/6, 0.0, 0.1], dtype=np.float32)
J_analytical = jacobian_scara(q_test)
J_newton = get_jacobian_from_newton_ik(model, ee_body_idx, q_test, mode="ANALYTIC")

print("Jacobian comparison (θ₂=30°):")
print(f"  Analytical shape: {J_analytical.shape}")
print(f"  Newton shape:     {J_newton.shape}")
print(f"  Max difference:   {np.max(np.abs(J_analytical - J_newton)):.6f}")
print()

# Sweep using analytical formula (fast)
for theta2 in theta2_range:
    q = np.array([0.0, theta2, 0.0, 0.1])
    J = jacobian_scara(q)
    JJT = J @ J.T
    manip = np.sqrt(max(0, np.linalg.det(JJT)))
    manipulability.append(manip)

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(np.degrees(theta2_range), manipulability, 'b-', linewidth=2)
ax.axvline(x=0, color='r', linestyle='--', linewidth=1.5, label='Singular (θ₂=0)')
ax.fill_between(np.degrees(theta2_range), manipulability, alpha=0.3)

ax.set_xlabel('θ₂ (degrees)', fontsize=12)
ax.set_ylabel('Manipulability √det(JJᵀ)', fontsize=12)
ax.set_title('Manipulability vs Elbow Angle (SCARA Robot)', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(-90, 90)

plt.tight_layout()
plt.savefig('manipulability.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ At θ₂=0 (fully extended), manipulability → 0 (singularity)")
print("✓ Maximum manipulability at θ₂=±90° (elbow bent)")
print("✓ Newton's ANALYTIC Jacobian matches the closed-form formula")


## Summary

### What We Covered

| Topic | Key Takeaway |
|-------|--------------|
| **Analytical IK** | Closed-form solution for simple robots (SCARA) |
| **Residuals** | IK minimizes ‖r‖² where r = [position, rotation, limits] |
| **IKPositionObjective** | 3 residuals (x,y,z) - reach target position |
| **IKRotationObjective** | 3 residuals - maintain end-effector orientation |
| **IKJointLimitObjective** | Prevent invalid joint configurations |
| **Batched IK** | Solve many problems in parallel (offline planning) |
| **Real-Time IK** | `set_target_position()` for dynamic updates (teleoperation) |
| **Singularities** | Detected via manipulability √det(JJᵀ) |

### Newton IK Quick Reference

```python
import newton

# Create objectives (combine as needed)
pos_obj = newton.ik.IKPositionObjective(link_index=ee_idx, target_positions=targets)
rot_obj = newton.ik.IKRotationObjective(link_index=ee_idx, target_rotations=rotations)
limit_obj = newton.ik.IKJointLimitObjective(joint_limit_lower=..., joint_limit_upper=...)

# Create solver
solver = newton.ik.IKSolver(
    model=model,
    n_problems=N,  # 1 for real-time, N for batched
    objectives=[pos_obj, rot_obj, limit_obj],
    jacobian_mode=newton.ik.IKJacobianMode.ANALYTIC,
)

# Batched solve (offline planning)
solver.step(joint_q_in, joint_q_out, iterations=50)

# Real-time solve (in control loop)
pos_obj.set_target_position(0, new_target)
solver.step(joint_q, joint_q, iterations=10)
```

### Further Reading
- `tutorial/02_inverse_kinematics.ipynb` - Franka robot IK example
- `newton.examples.ik_franka`, `newton.examples.ik_h1`


In [ ]:
print("✓ Tutorial 03: Inverse Kinematics - Complete!")
